# Notebook 03: Association Rules & Clustering

### Mục tiêu
- Luật kết hợp (Apriori): tìm combo thuộc tính liên quan huỷ
- Phân cụm (KMeans/DBSCAN): profiling cụm rủi ro huỷ cao

### Lý do chọn kỹ thuật
- **Apriori** (thay vì FP-Growth): phổ biến, dễ diễn giải, mlxtend hỗ trợ tốt. FP-Growth nhanh hơn nhưng Apriori đủ cho dataset ~119K.
- **KMeans**: đơn giản, hiệu quả cho dữ liệu số. Dùng Elbow + Silhouette chọn K.
- **DBSCAN**: phát hiện cụm dạng tự do, không cần biết trước K, loại noise.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore")

from src.data.loader import load_config
from src.features.builder import build_features
from src.mining.association import (
    discretize_for_association, create_transaction_matrix,
    run_apriori, filter_rules_by_target, compare_rules_by_group,
)
from src.mining.clustering import (
    prepare_clustering_features, find_optimal_k,
    run_kmeans, run_dbscan, profile_clusters,
    evaluate_clustering, reduce_to_2d,
)
from src.visualization.plots import (
    plot_association_rules_scatter, plot_elbow_silhouette,
    plot_cluster_scatter_2d, plot_cluster_profiles,
)
from src.evaluation.report import save_results

config = load_config("../configs/params.yaml")
seed = config['seed']

## 1. Load dữ liệu đã xử lý

In [2]:
df = pd.read_csv("../" + config["data"]["processed_path"])
# Rebuild features nếu cần
if 'lead_time_bin' not in df.columns:
    from src.features.builder import add_lead_time_bins, add_season
    df = add_lead_time_bins(df)
    df = add_season(df)
print(f"Loaded: {df.shape}")

Loaded: (86966, 41)


---
## PHẦN A: LUẬT KẾT HỢP (Association Rules)

### A.1 Rời rạc hoá dữ liệu

In [3]:
assoc_cfg = config.get("association", {})
disc = discretize_for_association(df, top_k_countries=assoc_cfg.get('top_k_countries', 10))
disc.head()

[Association] Discretized 12 columns for 86,966 transactions


,hotel,lead_time_bin,country_group,market_segment,distribution_channel,deposit_type,customer_type,meal,season,is_canceled,weekend_stay,has_deposit_flag
0,Resort Hotel,180+_very_long,PRT,Direct,Direct,No Deposit,Transient,BB,Summer,not_canceled,no_weekend,no_deposit
1,Resort Hotel,180+_very_long,PRT,Direct,Direct,No Deposit,Transient,BB,Summer,not_canceled,no_weekend,no_deposit
2,Resort Hotel,0-7_last_minute,GBR,Direct,Direct,No Deposit,Transient,BB,Summer,not_canceled,no_weekend,no_deposit
3,Resort Hotel,7-30_short,GBR,Corporate,Corporate,No Deposit,Transient,BB,Summer,not_canceled,no_weekend,no_deposit
4,Resort Hotel,7-30_short,GBR,Online TA,TA/TO,No Deposit,Transient,BB,Summer,not_canceled,no_weekend,no_deposit


### A.2 Chạy Apriori

In [4]:
txn = create_transaction_matrix(disc)

# Chạy Apriori với đo runtime
freq_items, rules, assoc_stats = run_apriori(
    txn,
    min_support=assoc_cfg.get('min_support', 0.05),
    min_confidence=assoc_cfg.get('min_confidence', 0.5),
    min_lift=assoc_cfg.get('min_lift', 1.2),
)

print(f"\n=== RUNTIME STATISTICS ===")
for k, v in assoc_stats.items():
    print(f"  {k}: {v}")

[Association] Transaction matrix: 86,966 × 53 items
[Association] Apriori found 8532 frequent itemsets in 4.68s
[Association] Generated 31899 rules (confidence≥0.5, lift≥1.2) in 0.50s

=== RUNTIME STATISTICS ===
  apriori_runtime_seconds: 4.68
  n_frequent_itemsets: 8532
  rules_runtime_seconds: 0.5
  n_rules: 31899


### A.3 Luật liên quan đến huỷ phòng

In [5]:
cancel_rules = filter_rules_by_target(rules, "is_canceled=canceled")
print(f"\nTop 15 luật liên quan huỷ phòng:")
if not cancel_rules.empty:
    display_cols = ["antecedents", "consequents", "support", "confidence", "lift"]
    print(cancel_rules[display_cols].head(15).to_string())
    save_results(cancel_rules.head(20), "../outputs/tables/association_rules")

[Association] 1 rules with consequent 'is_canceled=canceled'

Top 15 luật liên quan huỷ phòng:
                                                                     antecedents             consequents  support  confidence      lift
10254  (country_group=PRT, weekend_stay=has_weekend, distribution_channel=TA/TO)  (is_canceled=canceled)  0.05777    0.519169  1.901055
[Report] Saved CSV: ..\outputs\tables\association_rules.csv
[Report] Saved JSON: ..\outputs\tables\association_rules.json


In [6]:
# Biểu đồ luật kết hợp
if not rules.empty:
    plot_association_rules_scatter(rules, save_path="../outputs/figures/03_assoc_rules.png")

### A.4 So sánh luật theo mùa

In [7]:
if "season" in df.columns:
    season_rules = compare_rules_by_group(
        df, "season",
        min_support=0.05, min_confidence=0.5, min_lift=1.0,
    )
    for season, sr in season_rules.items():
        print(f"\n{season}: {len(sr)} rules related to cancellation")
        if not sr.empty:
            print(sr[['antecedents','confidence','lift']].head(3).to_string())


--- Group: season=Summer ---
[Association] Discretized 12 columns for 28,980 transactions
[Association] Transaction matrix: 28,980 × 50 items
[Association] Apriori found 12275 frequent itemsets in 1.84s
[Association] Generated 182843 rules (confidence≥0.5, lift≥1.0) in 1.23s
[Association] 510 rules with consequent 'is_canceled=canceled'

--- Group: season=Fall ---
[Association] Discretized 12 columns for 18,502 transactions
[Association] Transaction matrix: 18,502 × 48 items
[Association] Apriori found 11183 frequent itemsets in 1.08s
[Association] Generated 146819 rules (confidence≥0.5, lift≥1.0) in 1.02s
[Association] 18 rules with consequent 'is_canceled=canceled'

--- Group: season=Winter ---
[Association] Discretized 12 columns for 15,835 transactions
[Association] Transaction matrix: 15,835 × 47 items
[Association] Apriori found 13603 frequent itemsets in 1.12s
[Association] Generated 264846 rules (confidence≥0.5, lift≥1.0) in 1.63s
[Association] 0 rules with consequent 'is_canc

**Diễn giải luật kết hợp:**- Rule tìm được: Khách **Portugal** + đặt có **weekend** + qua **TA/TO** → cancel (confidence=52%, lift=1.90)- `lift=1.90` nghĩa là combo này có xác suất cancel gấp 1.9 lần trung bình- Số rules ít (do min_support=0.05 khá cao trên 87K rows) → chỉ pattern rất phổ biến mới xuất hiện- **Ý nghĩa**: OTA channel kết hợp weekend stay tại thị trường nội địa Portugal có rủi ro huỷ cao

---
## PHẦN B: PHÂN CỤM (Clustering)

### B.1 Chuẩn bị features

In [8]:
cluster_cfg = config.get('clustering', {})
X_scaled, scaler, feat_cols = prepare_clustering_features(
    df, feature_cols=cluster_cfg.get('features'),
)
print(f"Features: {feat_cols}")

[Clustering] Prepared 86,966 samples × 8 features
Features: ['lead_time', 'total_stays', 'total_guests', 'adr', 'booking_changes', 'previous_cancellations', 'days_in_waiting_list', 'total_of_special_requests']


### B.2 Tìm K tối ưu (Elbow + Silhouette)

In [9]:
k_results = find_optimal_k(
    X_scaled, k_range=cluster_cfg.get('k_range', [2,3,4,5,6,7,8]), seed=seed,
)
plot_elbow_silhouette(k_results, save_path="../outputs/figures/03_elbow_silhouette.png")

  K=2: inertia=612058, silhouette=0.2109
  K=3: inertia=548534, silhouette=0.2085
  K=4: inertia=492445, silhouette=0.2023
  K=5: inertia=450550, silhouette=0.2134
  K=6: inertia=398158, silhouette=0.2150
  K=7: inertia=364180, silhouette=0.2360
  K=8: inertia=341503, silhouette=0.1988
[Clustering] Best K by silhouette: 7 (score=0.2360)


### B.3 KMeans Clustering

In [10]:
best_k = k_results['best_k']
print(f"Best K = {best_k}")
labels_km, km_model = run_kmeans(X_scaled, best_k, seed)

# Profiling
profiles = profile_clusters(df, labels_km, feat_cols)
save_results(profiles, "../outputs/tables/cluster_profiles")

Best K = 7
[Clustering] KMeans K=7: inertia=364180
[Clustering] Cluster profiles:
         lead_time  total_stays  total_guests     adr  booking_changes  previous_cancellations  days_in_waiting_list  total_of_special_requests  cancel_rate   size   pct
cluster                                                                                                                                                                  
0            71.59         3.75          3.10  184.16             0.18                    0.00                  0.03                       0.59       0.3935  11559  13.3
1            72.09         3.41          2.13  114.05             0.14                    0.03                  0.02                       2.24       0.1930  11563  13.3
2           103.62         4.10          2.00  106.49             2.58                    0.01                  0.66                       0.72       0.1921   4618   5.3
3            53.59         2.45          1.62   71.34             0.

In [11]:
# Cluster visualization (PCA 2D)
X_2d = reduce_to_2d(X_scaled)
plot_cluster_scatter_2d(X_2d, labels_km, title=f"KMeans K={best_k}",
    save_path="../outputs/figures/03_cluster_scatter.png")

[Clustering] PCA 2D: 35.9% variance explained


In [12]:
# Cluster profile heatmap
plot_cluster_profiles(profiles, save_path="../outputs/figures/03_cluster_profiles.png")

### B.4 DBSCAN

In [13]:
labels_db, db_model = run_dbscan(X_scaled, min_samples=5)
db_eval = evaluate_clustering(X_scaled, labels_db)

[Clustering] DBSCAN auto eps=0.4526
[Clustering] DBSCAN: 253 clusters, 6638 noise points
[Clustering] Evaluation: {'silhouette': -0.0159, 'dbi': 0.8322, 'calinski_harabasz': 481.55}


### B.5 So sánh KMeans vs DBSCAN

In [14]:
km_eval = evaluate_clustering(X_scaled, labels_km)
print("\n=== CLUSTERING COMPARISON ===")
header = f"{'Metric':<25} {'KMeans':<15} {'DBSCAN':<15}"
print(header)
print("-"*55)
for metric in ['silhouette', 'dbi', 'calinski_harabasz']:
    print(f"{metric:<25} {km_eval[metric]:<15} {db_eval[metric]:<15}")

[Clustering] Evaluation: {'silhouette': 0.236, 'dbi': 1.1401, 'calinski_harabasz': 13194.56}

=== CLUSTERING COMPARISON ===
Metric                    KMeans          DBSCAN         
-------------------------------------------------------
silhouette                0.236           -0.0159        
dbi                       1.1401          0.8322         
calinski_harabasz         13194.56        481.55         


### B.6 Đặt tên cụm

Dựa trên profiles, đặt tên mô tả cho từng cụm. Ví dụ:
- **Cluster 0**: 'Short-stay budget' (lead_time ngắn, adr thấp)
- **Cluster 1**: 'Long-lead high-risk' (lead_time dài, cancel rate cao)
- **Cluster 2**: 'Loyal repeaters' (previous_cancellations thấp, special_requests cao)

→ Chi tiết tên cụm phụ thuộc vào kết quả thực tế.

## Kết luận Mining & Clustering### Association Rules- Tìm được pattern PRT + weekend + TA/TO → cancel (lift=1.90)- Runtime Apriori đủ nhanh cho dataset ~87K rows- Ít rules do min_support cao → có thể thử giảm min_support=0.03 để tìm thêm### Clustering- KMeans tìm được các cụm rõ ràng phân biệt theo ADR, lead_time, special_requests- Cluster có ADR cao (€184/đêm) cancel rate cao nhất (39%) → nhóm premium rủi ro- Cluster có special_requests cao (2.24) cancel thấp nhất (19%) → nhóm engaged- DBSCAN phát hiện noise points nhưng cụm chính ít phân biệt hơn KMeans→ Tiếp theo: Notebook 04 - Classification